<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [1]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import pandas as pd
from sklearn.datasets import load_wine
import collections
# 데이터 로드 및 데이터프레임 생성
wine_dataset = load_wine()
wine_dataframe = pd.DataFrame(wine_dataset.data, columns=wine_dataset.feature_names)
wine_dataframe["quality"] = wine_dataset.target

</br>

# 학습 내용
>이번 장에서는 <strong>전처리 및 모델 학습(Preprocessing & Model Training)</strong>에 대해 학습합니다.</br></br>
>스케일링, 데이터 분할, 로지스틱 회귀 모델 학습의 전체 파이프라인을 학습해봅시다.

</br>

In [50]:
# TODO 1:
#  - features에 quality를 제외한 데이터를 저장해봅시다.
#  - target에 quality 데이터를 저장해봅시다.
features = wine_dataframe.drop(columns=['quality'])
target = wine_dataframe['quality']

print("--- [TODO 1] Features (문제지) 크기 ---")
print(features.shape)
print("\n--- [TODO 1] Target (정답지) 크기 ---")
print(target.shape)

--- [TODO 1] Features (문제지) 크기 ---
(178, 13)

--- [TODO 1] Target (정답지) 크기 ---
(178,)


In [49]:
# TODO 2: train_test_split을 import 해봅시다.
from sklearn.model_selection import train_test_split

In [48]:
# TODO 3: features, target을 학습용과 테스트용으로 분할해봅시다. (test_size=0.2, random_state=42, stratify=target)
# test_size=0.2 (20%를 테스트용으로), stratify=target (정답 비율 유지)
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

In [47]:
# TODO 5: 분할된 데이터의 클래스 비율이 유지되는지 확인해봅시다.
# normalize=True를 사용하면 개수가 아닌 0~1 사이의 비율(%)로 출력됩니다.
print("--- [TODO 5] 학습용 데이터(Train)의 클래스 비율 ---")
print(target_train.value_counts(normalize=True).sort_index())

print("\n--- [TODO 5] 테스트용 데이터(Test)의 클래스 비율 ---")
print(target_test.value_counts(normalize=True).sort_index())

--- [TODO 5] 학습용 데이터(Train)의 클래스 비율 ---
quality
0    0.330986
1    0.401408
2    0.267606
Name: proportion, dtype: float64

--- [TODO 5] 테스트용 데이터(Test)의 클래스 비율 ---
quality
0    0.333333
1    0.388889
2    0.277778
Name: proportion, dtype: float64


💡stratify 파라미터
> 분류 문제에서 `stratify=y`를 지정하면 학습/테스트 세트의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">클래스 비율이 원본과 동일</mark>하게 유지됩니다.

</br>

# 피처 스케일링 (Feature Scaling)
> 피처의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">값 범위를 통일</mark>하여 모델 학습을 안정화하는 전처리 과정입니다.

> 전처리(Preprocessing)는 원시 데이터를 모델이 학습할 수 있는 형태로 변환하는 과정으로, 이 단계를 건너뛰면 알고리즘이 데이터의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">진짜 패턴</mark> 대신 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">형식적 특성</mark>을 학습합니다.</br></br>
> 예를 들어 `나이(20~80)`와 `연봉(1,000만~1억)` 피처가 함께 있을 때 스케일링 없이 학습하면, KNN·SVM·신경망 등 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">거리·경사 기반 알고리즘</mark>은 수치가 큰 피처에 과도하게 의존하게 됩니다.</br></br>
> 머신러닝 파이프라인은 `원시 데이터 → EDA → 결측치/이상치 처리 → 피처 엔지니어링 → 스케일링 → 데이터 분할 → 모델 학습 → 평가`의 흐름을 따르며, 특히 스케일링은 반드시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">데이터 분할 이후 학습 데이터에만 fit</mark>해야 테스트 데이터 누수를 막을 수 있습니다.

In [46]:
# TODO 6: StandardScaler를 import 해봅시다.
from sklearn.preprocessing import StandardScaler

In [45]:
# TODO 7: 학습 데이터에 StandardScaler를 학습(fit)하고 변환(transform)해봅시다.
scaler = StandardScaler()
features_train_scaled = scaler.fit_transform(features_train)

In [44]:
# TODO 8: 테스트 데이터에는 변환(transform)만 적용해봅시다.
features_test_scaled = scaler.transform(features_test)

print("--- 스케일링 완료 (Train/Test 데이터) ---")
print(f"Train_scaled 평균: {features_train_scaled.mean():.2f}, 표준편차: {features_train_scaled.std():.2f}")

--- 스케일링 완료 (Train/Test 데이터) ---
Train_scaled 평균: -0.00, 표준편차: 1.00


💡스케일링 주의사항
> `fit_transform`은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 데이터에만</mark> 적용합니다.</br>
> 테스트 데이터에는 `transform`만 사용하여 데이터 누수(Data Leakage)를 방지합니다.

</br>

In [43]:
# TODO 9: LogisticRegression을 import 해봅시다.
from sklearn.linear_model import LogisticRegression

In [42]:
# TODO 10: 로지스틱 회귀 모델(max_iter=5000, random_state=42)을 생성해봅시다.
# max_iter를 넉넉히 주어 모델이 최적점을 찾을 때까지 충분히 반복하도록 합니다.
model = LogisticRegression(max_iter=5000, random_state=42)

💡`LogisticRegression` 주요 파라미터
> <table style="width:100%">
>   <thead>
>     <tr>
>       <th style="text-align:center">파라미터</th>
>       <th style="text-align:center">설명</th>
>       <th style="text-align:center">예시</th>
>     </tr>
>   </thead>
>   <tbody>
>     <tr><td style="text-align:center"><code>max_iter</code></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">maximum iterations</mark>의 약어. 최적의 파라미터를 찾기 위한 최대 반복 횟수</td><td style="text-align:center">기본값 100, 수렴하지 않으면 늘림</td></tr>
>     <tr><td style="text-align:center"><code>random_state</code></td><td style="text-align:center">난수 생성 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">시드(seed)</mark> 값. 같은 값을 지정하면 실행할 때마다 동일한 결과 보장</td><td style="text-align:center">42는 관례적으로 자주 사용</td></tr>
>   </tbody>
> </table></br>
> `ConvergenceWarning`이 발생하면 모델이 최적 해를 찾지 못한 것이므로 `max_iter`를 늘려야 합니다.

In [33]:
# TODO 11: 스케일링된 학습 데이터로 모델을 학습시켜봅시다.
model.fit(features_train_scaled, target_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [40]:
# TODO 12: 스케일링된 테스트 데이터로 예측해봅시다.
pred = model.predict(features_test_scaled)
print("--- 예측 결과 vs 실제 정답 (상위 10개) ---")
print(f"실제 정답 (target_test): {target_test[:10].values}")
print(f"모델 예측 (pred):        {pred[:10]}")

--- 예측 결과 vs 실제 정답 (상위 10개) ---
실제 정답 (target_test): [0 2 0 1 1 0 0 1 1 2]
모델 예측 (pred):        [0 1 0 1 1 0 0 1 1 2]


In [41]:
# TODO 13: 테스트 데이터에 대한 정확도를 출력해봅시다.
accuracy = model.score(features_test_scaled, target_test)
print(f"--- 최종 모델 정확도 (Accuracy): {accuracy * 100:.2f}% ---")

--- 최종 모델 정확도 (Accuracy): 97.22% ---


</br>

## scikit-learn 학습 패턴
> 모든 scikit-learn 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 API 패턴</mark>을 따릅니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">메서드</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>model.fit(X, y)</code></td><td>학습 데이터로 모델 학습</td></tr>
    <tr><td style="text-align:center"><code>model.predict(X)</code></td><td>새 데이터에 대한 예측</td></tr>
    <tr><td style="text-align:center"><code>model.score(X, y)</code></td><td>정확도 (분류) / R² (회귀)</td></tr>
    <tr><td style="text-align:center"><code>model.predict_proba(X)</code></td><td>클래스별 확률 예측</td></tr>
  </tbody>
</table>